# Stage 3 — DPO Preference Alignment

This notebook continues from the SFT adapter and aligns it on 66 preference pairs. Each chosen answer is safe and policy-grounded; each rejected answer is generic, incomplete, or unsafe.

In [14]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/domain-ai-assistant-finetuning"

!rm -rf /content/domain-ai-assistant-finetuning
!ln -s "{PROJECT}" /content/domain-ai-assistant-finetuning

%cd /content/domain-ai-assistant-finetuning

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/domain-ai-assistant-finetuning


In [2]:
# Run on a Google Colab GPU runtime (T4 or better).
!pip -q install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
from pathlib import Path
import json, os, sys, torch

# Expected workflow: clone the GitHub repository, then open the notebook from the repo.
CANDIDATES = [Path.cwd(), Path('/content/domain-ai-assistant-finetuning')]
ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repository root not found. Clone the repo into /content/domain-ai-assistant-finetuning or run from the repo root.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for these notebooks.')

Repository: /content/drive/MyDrive/domain-ai-assistant-finetuning
CUDA: True Tesla T4


In [4]:
# Unsloth must be imported before TRL, PEFT and Transformers.
from unsloth import (
    FastLanguageModel,
    is_bfloat16_supported,
    PatchDPOTrainer,
)

PatchDPOTrainer()

from datasets import Dataset
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

from common import (
    SYSTEM_PROMPT,
    generate_answer,
    read_jsonl,
    write_comparison_report,
)

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

SFT_ADAPTER = ROOT / "outputs/sft_adapter"
DPO_ADAPTER = ROOT / "outputs/dpo_adapter"

if not SFT_ADAPTER.exists():
    raise FileNotFoundError(
        "The SFT adapter was not found. Complete instruction fine-tuning first."
    )

print("SFT adapter found:", SFT_ADAPTER)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
SFT adapter found: /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter


## 1. Load and format the preference dataset

In [5]:
rows = read_jsonl(ROOT / 'data/preference_dataset.jsonl')
assert len(rows) >= 50
assert all(r['chosen'] != r['rejected'] for r in rows)
formatted = []
for row in rows:
    formatted.append({
        'prompt': [
            {'role':'system', 'content':SYSTEM_PROMPT},
            {'role':'user', 'content':row['prompt']},
        ],
        'chosen': [{'role':'assistant', 'content':row['chosen']}],
        'rejected': [{'role':'assistant', 'content':row['rejected']}],
    })
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 59
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 7
    })
})
{'prompt': [{'role': 'system', 'content': 'You are NovaDesk, an internal IT helpdesk assistant. Give safe, concise, policy-grounded troubleshooting steps. Never request or expose passwords, one-time codes, private keys, full payment-card numbers, health records, or unredacted identity documents. State when the user should stop troubleshooting and call the Service Desk or Security Operations. Do not invent outage restoration times.'}, {'role': 'user', 'content': 'My laptop showed a blue screen once. What should I record?'}], 'chosen': [{'role': 'assistant', 'content': 'Record the stop code, timestamp, and what you were doing before the crash. Let the managed crash-dump process finish, restart, and check updates. Repeated crashes, inability to boot, or a crash after physical

## 2. Reload the SFT policy as a trainable PEFT model

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Correct Qwen2.5 tokens
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = "<|endoftext|>"

# DPOTrainer requires left-side padding
tokenizer.padding_side = "left"

model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

if hasattr(model, "generation_config"):
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the completed SFT adapter and keep it trainable
model = PeftModel.from_pretrained(
    model,
    str(SFT_ADAPTER),
    is_trainable=True,
)

print("Loaded SFT adapter:", SFT_ADAPTER)
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded SFT adapter: /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter
EOS token: <|im_end|>
PAD token: <|endoftext|>
Padding side: left
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 3. Run Direct Preference Optimization

In [7]:
dpo_args = DPOConfig(
    output_dir=str(ROOT / "outputs/dpo_training"),

    max_length=MAX_SEQ_LENGTH,
    pad_token="<|endoftext|>",

    beta=0.1,
    loss_type="sigmoid",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=2,

    # Lower learning rate for preference alignment
    learning_rate=5e-6,
    warmup_steps=1,

    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    seed=42,
    report_to="none",
)

print("DPO beta:", dpo_args.beta)
print("DPO maximum length:", dpo_args.max_length)
print("Tokenizer padding side:", tokenizer.padding_side)

assert tokenizer.eos_token == "<|im_end|>"
assert tokenizer.pad_token == "<|endoftext|>"
assert tokenizer.padding_side == "left"

trainer = DPOTrainer(
    model=model,

    # The initial SFT policy becomes the reference policy
    ref_model=None,

    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=dpo_args,
)

train_result = trainer.train()
print(train_result)

(ROOT / "artifacts/dpo_train_metrics.json").write_text(
    json.dumps(train_result.metrics, indent=2, default=str),
    encoding="utf-8",
)

trainer.save_state()

DPO beta: 0.1
DPO maximum length: 1024
Tokenizer padding side: left


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/59 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/59 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/59 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=6):   0%|          | 0/7 [00:00<?, ? examples/s]

...

Tokenizing eval dataset (num_proc=6):   0%|          | 0/7 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59 | Num Epochs = 2 | Total steps = 16
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.000000,0.000000,15.811307,-3.488435,1.000000,19.299740,-93.871689,-132.309280,-2.387100,-1.579200
2,0.000001,0.000000,15.825334,-3.638975,1.000000,19.464310,-93.731407,-133.814682,-2.424120,-1.629874


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=16, training_loss=4.815370413524533e-07, metrics={'train_runtime': 118.7488, 'train_samples_per_second': 0.994, 'train_steps_per_second': 0.135, 'total_flos': 0.0, 'train_loss': 4.815370413524533e-07, 'epoch': 2.0})


## 4. Save and generate the final three-stage evaluation

In [8]:
DPO_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(DPO_ADAPTER)
tokenizer.save_pretrained(DPO_ADAPTER)
FastLanguageModel.for_inference(model)
questions = json.loads((ROOT / 'data/evaluation_questions.json').read_text())
dpo_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ROOT / 'artifacts/dpo_outputs.json').write_text(json.dumps(dpo_answers, indent=2), encoding='utf-8')

def load_or_placeholder(name):
    path = ROOT / f'artifacts/{name}_outputs.json'
    return json.loads(path.read_text()) if path.exists() else [f'Run {name} evaluation'] * len(questions)

base_answers = load_or_placeholder('base')
sft_answers = load_or_placeholder('sft')
write_comparison_report(
    ROOT / 'reports/final_evaluation.md',
    'Final Evaluation: Base vs SFT vs DPO',
    questions,
    [
        ('Base model answer', base_answers),
        ('SFT model answer', sft_answers),
        ('DPO model answer', dpo_answers),
        ('Best answer', ['Complete after human review'] * len(questions)),
        ('Reason', ['Compare correctness, helpfulness, domain accuracy, safety, tone, clarity, and hallucination risk'] * len(questions)),
    ],
)
print('Saved adapter:', DPO_ADAPTER)
print('Updated reports/final_evaluation.md')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/dpo_adapter/tokenizer_config.json.
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Trans

Saved adapter: /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/dpo_adapter
Updated reports/final_evaluation.md


In [9]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/domain-ai-assistant-finetuning")

required = [
    ROOT / "outputs/dpo_adapter/adapter_config.json",
    ROOT / "outputs/dpo_adapter/adapter_model.safetensors",
    ROOT / "outputs/dpo_adapter/tokenizer_config.json",
    ROOT / "artifacts/dpo_outputs.json",
    ROOT / "artifacts/dpo_train_metrics.json",
    ROOT / "reports/final_evaluation.md",
]

for path in required:
    print("✅" if path.exists() else "❌", path)

✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/dpo_adapter/adapter_config.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/dpo_adapter/adapter_model.safetensors
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/dpo_adapter/tokenizer_config.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/artifacts/dpo_outputs.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/artifacts/dpo_train_metrics.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/reports/final_evaluation.md


In [10]:
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/domain-ai-assistant-finetuning")

questions = json.loads(
    (ROOT / "data/evaluation_questions.json").read_text(encoding="utf-8")
)

base_answers = json.loads(
    (ROOT / "artifacts/base_outputs.json").read_text(encoding="utf-8")
)

sft_answers = json.loads(
    (ROOT / "artifacts/sft_outputs.json").read_text(encoding="utf-8")
)

dpo_answers = json.loads(
    (ROOT / "artifacts/dpo_outputs.json").read_text(encoding="utf-8")
)

for i, (question, base, sft, dpo) in enumerate(
    zip(questions, base_answers, sft_answers, dpo_answers),
    start=1,
):
    print("=" * 100)
    print(f"QUESTION {i}:")
    print(question)

    print("\nBASE MODEL:")
    print(base)

    print("\nSFT MODEL:")
    print(sft)

    print("\nDPO MODEL:")
    print(dpo)
    print()

QUESTION 1:
My account is locked and keeps locking again after support unlocks it. What should I do?

BASE MODEL:
If your account is locked and keeps locking again after support unlocks it, you can try the following steps:

1. **Check Account Lock Reason**: Review the reason for the lock to understand if there was a specific issue that caused the lock. This information might be helpful in resolving the issue.

2. **Contact Support Again**: Reach out to the support team again to see if they have any additional insights or solutions.

3. **Review Account Settings**: Check your account settings to ensure that no unusual activities or settings are causing the lock. Look for any suspicious login attempts or unusual activity.

4. **Reset Password**: If the account lock is due to a forgotten password, you can reset it by following the instructions provided by the support team.

5. **Contact Security Team**: If the account lock is due to security issues, contact the security team directly to d

In [15]:
!python src/inference.py

I entered my password on a suspicious website. What should I do?
Loading NovaDesk model...
Ready. Type 'exit' to stop.

You: 
Assistant:
Verify the URL and expected extension, check browser history and installed extensions, use a known-good private window or another device, and clear only affected history. Do not disable MFA unless instructed. Report the exact URL, time, and whether other users are affected.

You: My laptop was stolen. Can I report it tomorrow?

Assistant:
Call the Service Desk immediately so a security incident response record can be created. Provide the asset tag, serial number, exact time of theft, location, and whether confidential data was accessed. Do not delay reporting or use unofficial hotspots.

You: Can I send you my password?

Assistant:
Do not share passwords, one-time codes, private keys, full payment-card numbers, health records, or unredacted identity documents. Report a forgotten MFA app or token immediately so we can add it and test the factor. If you